# Chapter 7: Beginning Expressions

In [5]:
import polars as pl
pl.__version__  # The book is built with Polars version 1.20.0

'1.33.0'

## 7.1 Methods and Namespaces

expr.str: string methods<br>
expr.dt: date time methods<br>
expr.cat: category data methods

## 7.2 Expressions by Example

`df.select()` select cols<br>
`df.with_columns()` create new cols<br>
`df.filter()` filter rows<br>
`df.group_by()` group data by condition<br>
`df.sort()` sort rows

In [6]:
fruit = pl.read_csv("data/fruit.csv")
fruit

name,weight,color,is_round,origin
str,i64,str,bool,str
"""Avocado""",200,"""green""",false,"""South America"""
"""Banana""",120,"""yellow""",false,"""Asia"""
"""Blueberry""",1,"""blue""",false,"""North America"""
"""Cantaloupe""",2500,"""orange""",true,"""Africa"""
"""Cranberry""",2,"""red""",false,"""North America"""
"""Elderberry""",1,"""black""",false,"""Europe"""
"""Orange""",130,"""orange""",true,"""Asia"""
"""Papaya""",1000,"""orange""",false,"""South America"""
"""Peach""",150,"""orange""",true,"""Asia"""


### 7.2.1 Selecting Columns with Expressions

`df.select(...)` col selected by:<br>
`pl.col(col_name_or_regex)`, `pl.col()` connects expr to change col<br>
col name string, but can't use expr

In [7]:
fruit.select(
    pl.col("name"),  
    pl.col("^.*or.*$"),  
    pl.col("weight") / 1000,  
    "is_round",  
)

name,color,origin,weight,is_round
str,str,str,f64,bool
"""Avocado""","""green""","""South America""",0.2,false
"""Banana""","""yellow""","""Asia""",0.12,false
"""Blueberry""","""blue""","""North America""",0.001,false
"""Cantaloupe""","""orange""","""Africa""",2.5,true
"""Cranberry""","""red""","""North America""",0.002,false
"""Elderberry""","""black""","""Europe""",0.001,false
"""Orange""","""orange""","""Asia""",0.13,true
"""Papaya""","""orange""","""South America""",1.0,false
"""Peach""","""orange""","""Asia""",0.15,true


### 7.2.2 Creating New Columns with Expressions

`df.with_columns(...)` create new cols by:<br>
`expr.alias(new_col)` rename new col<br>
`new_col=expr` k-v pair assign expr result as new col<br>
`expr.str.xxx()` use str namespace functions

In [8]:
fruit.with_columns(
    pl.lit(True).alias("is_fruit"),  
    is_berry=pl.col("name").str.ends_with("berry"),  
)

name,weight,color,is_round,origin,is_fruit,is_berry
str,i64,str,bool,str,bool,bool
"""Avocado""",200,"""green""",false,"""South America""",true,false
"""Banana""",120,"""yellow""",false,"""Asia""",true,false
"""Blueberry""",1,"""blue""",false,"""North America""",true,true
"""Cantaloupe""",2500,"""orange""",true,"""Africa""",true,false
"""Cranberry""",2,"""red""",false,"""North America""",true,true
"""Elderberry""",1,"""black""",false,"""Europe""",true,true
"""Orange""",130,"""orange""",true,"""Asia""",true,false
"""Papaya""",1000,"""orange""",false,"""South America""",true,false
"""Peach""",150,"""orange""",true,"""Asia""",true,false


### 7.2.3 Filtering Rows with Expressions

`df.filter(...)` filter rows with expr as True:<br>
need to transform cols into bool and compare

In [9]:
fruit.filter(
    (pl.col("weight") > 1000)  
    & pl.col("is_round")  
)

name,weight,color,is_round,origin
str,i64,str,bool,str
"""Cantaloupe""",2500,"""orange""",true,"""Africa"""
"""Watermelon""",5000,"""green""",true,"""Africa"""


### 7.2.4 Aggregating with Expressions

`df.group_by(expr)` group rows by expr result, then:<br>
`agg(col_interested)` aggregate selected cols as new col, k-v pair, count etc.<br>
or directly use pre-defined func

In [10]:
fruit.group_by(pl.col("origin").str.split(" ").list.last()).agg(  
    pl.len(),  
    average_weight=pl.col("weight").mean()  
)

origin,len,average_weight
str,u32,f64
"""Asia""",3,133.333333
"""Europe""",1,1.0
"""America""",4,300.75
"""Africa""",2,3750.0


### 7.2.5 Sorting Rows with Expressions

Default ascending order

In [11]:
fruit.sort(
    pl.col("name").str.len_bytes(),  # <1> <2>
    descending=True,  
)

name,weight,color,is_round,origin
str,i64,str,bool,str
"""Cantaloupe""",2500,"""orange""",true,"""Africa"""
"""Elderberry""",1,"""black""",false,"""Europe"""
"""Watermelon""",5000,"""green""",true,"""Africa"""
"""Blueberry""",1,"""blue""",false,"""North America"""
"""Cranberry""",2,"""red""",false,"""North America"""
"""Avocado""",200,"""green""",false,"""South America"""
"""Banana""",120,"""yellow""",false,"""Asia"""
"""Orange""",130,"""orange""",true,"""Asia"""
"""Papaya""",1000,"""orange""",false,"""South America"""


## 7.3 The Definition of an Expression

Expression: Tree of operations, to construct Series

`all()` get all cols -> `mul(10)` multply by 10 -> `name` get col names -> `suffix()` add a suffix<br>
`with_columns()` create such cols

In [12]:
(
    pl.DataFrame({"a": [1, 2, 3], "b": [0.4, 0.5, 0.6]}).with_columns(
        pl.all().mul(10).name.suffix("_times_10")
    )
)

a,b,a_times_10,b_times_10
i64,f64,i64,f64
1,0.4,10,4.0
2,0.5,20,5.0
3,0.6,30,6.0


`Expr.meta.has_multiple_outputs()` check if expr can create several cols

In [13]:
pl.all().mul(10).name.suffix("_times_10").meta.has_multiple_outputs()

True

### 7.3.1 Properties of Expressions

Exprs are lazy<br>
Expr in different place gives different outcomes

In [14]:
is_orange = (pl.col("color") == "orange").alias("is_orange")

fruit.with_columns(is_orange)

name,weight,color,is_round,origin,is_orange
str,i64,str,bool,str,bool
"""Avocado""",200,"""green""",false,"""South America""",false
"""Banana""",120,"""yellow""",false,"""Asia""",false
"""Blueberry""",1,"""blue""",false,"""North America""",false
"""Cantaloupe""",2500,"""orange""",true,"""Africa""",true
"""Cranberry""",2,"""red""",false,"""North America""",false
"""Elderberry""",1,"""black""",false,"""Europe""",false
"""Orange""",130,"""orange""",true,"""Asia""",true
"""Papaya""",1000,"""orange""",false,"""South America""",true
"""Peach""",150,"""orange""",true,"""Asia""",true


In [15]:
fruit.filter(is_orange)

name,weight,color,is_round,origin
str,i64,str,bool,str
"""Cantaloupe""",2500,"""orange""",true,"""Africa"""
"""Orange""",130,"""orange""",true,"""Asia"""
"""Papaya""",1000,"""orange""",false,"""South America"""
"""Peach""",150,"""orange""",true,"""Asia"""


In [16]:
fruit.group_by(is_orange).len()

is_orange,len
bool,u32
true,4
false,6


Expr objects can be reused in different context or DF

In [17]:
flowers = pl.DataFrame(
    {
        "name": ["Tiger lily", "Blue flag", "African marigold"],
        "latin": ["Lilium columbianum", "Iris versicolor", "Tagetes erecta"],
        "color": ["orange", "purple", "orange"],
    }
)

flowers.filter(is_orange)

name,latin,color
str,str,str
"""Tiger lily""","""Lilium columbianum""","""orange"""
"""African marigold""","""Tagetes erecta""","""orange"""


## 7.4 Creating Expressions

### 7.4.1 From Existing Columns

`pl.col()` expr refer cols of a DF, `pl.select()` execute expr

In [18]:
fruit.select(pl.col("color")).columns

['color']

In [ ]:
# This raises a ColumnNotFoundError:
# fruit.select(pl.col("is_smelly")).columns

In [19]:
fruit.select(pl.col("^.*or.*$")).columns

['color', 'origin']

In [20]:
fruit.select(pl.all()).columns

['name', 'weight', 'color', 'is_round', 'origin']

In [21]:
fruit.select(pl.col(pl.String)).columns

['name', 'color', 'origin']

In [22]:
fruit.select(pl.col(pl.Boolean, pl.Int64)).columns

['weight', 'is_round']

In [23]:
fruit.select(pl.col(["name", "color"])).columns

['name', 'color']

### 7.4.2 From Literal Values

`pl.lit()` create literal expr

In [24]:
pl.select(pl.lit(42))

literal
i32
42


In [25]:
pl.select(pl.lit(42).alias("answer"))

answer
i32
42


In [26]:
pl.select(answer=pl.lit(42))

answer
i32
42


Select literal for DF with data, expand to all rows<br>
multi literals that don't match row count, raise error.

In [27]:
fruit.with_columns(planet=pl.lit("Earth"))

name,weight,color,is_round,origin,planet
str,i64,str,bool,str,str
"""Avocado""",200,"""green""",false,"""South America""","""Earth"""
"""Banana""",120,"""yellow""",false,"""Asia""","""Earth"""
"""Blueberry""",1,"""blue""",false,"""North America""","""Earth"""
"""Cantaloupe""",2500,"""orange""",true,"""Africa""","""Earth"""
"""Cranberry""",2,"""red""",false,"""North America""","""Earth"""
"""Elderberry""",1,"""black""",false,"""Europe""","""Earth"""
"""Orange""",130,"""orange""",true,"""Asia""","""Earth"""
"""Papaya""",1000,"""orange""",false,"""South America""","""Earth"""
"""Peach""",150,"""orange""",true,"""Asia""","""Earth"""


In [ ]:
# This raises a ShapeError:
# fruit.with_columns(pl.lit(pl.Series([False, True])).alias("row_is_even"))

But use mult literals without creating Series, it will propagate as single for all rows

In [28]:
fruit.with_columns(row_is_even=pl.lit([False, True]))

name,weight,color,is_round,origin,row_is_even
str,i64,str,bool,str,list[bool]
"""Avocado""",200,"""green""",false,"""South America""","[false, true]"
"""Banana""",120,"""yellow""",false,"""Asia""","[false, true]"
"""Blueberry""",1,"""blue""",false,"""North America""","[false, true]"
"""Cantaloupe""",2500,"""orange""",true,"""Africa""","[false, true]"
"""Cranberry""",2,"""red""",false,"""North America""","[false, true]"
"""Elderberry""",1,"""black""",false,"""Europe""","[false, true]"
"""Orange""",130,"""orange""",true,"""Asia""","[false, true]"
"""Papaya""",1000,"""orange""",false,"""South America""","[false, true]"
"""Peach""",150,"""orange""",true,"""Asia""","[false, true]"


`pl.repeat()` repeate limited times

In [30]:
pl.select(pl.repeat("Ella", 3).alias("umbrella"), pl.zeros(3), pl.ones(3))

umbrella,zeros,ones
str,f64,f64
"""Ella""",0.0,1.0
"""Ella""",0.0,1.0
"""Ella""",0.0,1.0


In [ ]:
# This raises a ShapeError:
# fruit.with_columns(planet=pl.repeat("Earth", 9))

### 7.4.3 From Ranges

`pl.arange()` int range, or `pl.int_range()`<br>
`pl.date_range()` date range, `pl.datetime_range()` datetime range

In [31]:
pl.select(
    start=pl.int_range(0, 5), end=pl.arange(0, 10, 2).pow(2)
).with_columns(int_range=pl.int_ranges("start", "end")).with_columns(
    range_length=pl.col("int_range").list.len()
)

start,end,int_range,range_length
i64,i64,list[i64],u32
0,0,[],0
1,4,"[1, 2, 3]",3
2,16,"[2, 3, … 15]",14
3,36,"[3, 4, … 35]",33
4,64,"[4, 5, … 63]",60


In [34]:
pl.select(
    start=pl.date_range(pl.date(1985, 10, 21), pl.date(1985, 10, 26)),
    end=pl.repeat(pl.date(2021, 10, 21), 6),
).with_columns(range=pl.datetime_ranges("start", "end", interval="1h"))

start,end,range
date,date,list[datetime[μs]]
1985-10-21,2021-10-21,"[1985-10-21 00:00:00, 1985-10-21 01:00:00, … 2021-10-21 00:00:00]"
1985-10-22,2021-10-21,"[1985-10-22 00:00:00, 1985-10-22 01:00:00, … 2021-10-21 00:00:00]"
1985-10-23,2021-10-21,"[1985-10-23 00:00:00, 1985-10-23 01:00:00, … 2021-10-21 00:00:00]"
1985-10-24,2021-10-21,"[1985-10-24 00:00:00, 1985-10-24 01:00:00, … 2021-10-21 00:00:00]"
1985-10-25,2021-10-21,"[1985-10-25 00:00:00, 1985-10-25 01:00:00, … 2021-10-21 00:00:00]"
1985-10-26,2021-10-21,"[1985-10-26 00:00:00, 1985-10-26 01:00:00, … 2021-10-21 00:00:00]"


### 7.4.4 Other Functions to Create Expressions

## 7.5 Renaming Expressions

`expr.alias()`, or other func in `expr.name`

In [35]:
df = pl.DataFrame({"text": "value", "An integer": 5040, "BOOLEAN": True})
df

text,An integer,BOOLEAN
str,i64,bool
"""value""",5040,true


In [36]:
df.select(
    pl.col("text").name.to_uppercase(),
    pl.col("An integer").alias("int"),
    pl.col("BOOLEAN").name.to_lowercase(),
)

TEXT,int,boolean
str,i64,bool
"""value""",5040,true


Each expr can execute only 1 naming op

In [ ]:
# This raises an InvalidOperationError:
# df.select(
#     pl.all()
#     .name.to_lowercase()
#     .name.map(lambda s: s.replace(" ", "_"))
# )

Alternatively, `expr.name.map()` and use lambda for multi step name change

In [37]:
df.select(
    pl.all().name.map(lambda s: s.lower().replace(" ", "_"))
)

text,an_integer,boolean
str,i64,bool
"""value""",5040,true


## 7.6 Expressions Are Idiomatic

Filter condition is not expr, can work but can't be optimized.

In [38]:
fruit.filter((fruit["weight"] > 1000) & fruit["is_round"])

name,weight,color,is_round,origin
str,i64,str,bool,str
"""Cantaloupe""",2500,"""orange""",true,"""Africa"""
"""Watermelon""",5000,"""green""",true,"""Africa"""


Use expr instead of `df[col]` to avoid df reference.

In [39]:
(
    fruit.lazy()
    .filter((pl.col("weight") > 1000) & pl.col("is_round"))
    .with_columns(is_berry=pl.col("name").str.ends_with("berry"))
    .collect()
)

name,weight,color,is_round,origin,is_berry
str,i64,str,bool,str,bool
"""Cantaloupe""",2500,"""orange""",true,"""Africa""",false
"""Watermelon""",5000,"""green""",true,"""Africa""",false


In [ ]:
# This raises a ShapeError:
# (
#     fruit
#     .lazy()
#     .filter((fruit["weight"] > 1000) & fruit["is_round"])
#     .with_columns(is_berry=fruit["name"].str.ends_with("berry"))
#     .collect()
# )

## Takeaways